# Random Forest com média e desvio padrão

Testamos uma floresta de árvores para classificar controle ou diabetes. Cada árvore recebe uma amostra sorteada com reposição do treinamento. Em cada nó, comparamos um subconjunto sorteado de atributos. Ao final, combinamos as respostas das árvores.

Implementamos as contas com NumPy. A floresta é uma extensão do trabalho de árvores por solicitação do aluno. As referências das aulas sustentam as árvores e a avaliação; o sorteio de amostras e atributos da floresta segue a referência técnica indicada ao final. As escolhas de implementação e a grade de comparação ficam declaradas aqui.

Usamos a fórmula `(valor − média) / desvio padrão` nas entradas. A média e o desvio são calculados somente na parte usada para treinar. A reserva e o teste recebem esses mesmos parâmetros.

### Resultado desta execução

A busca inicial comparou 768 configurações. A segunda seleção incluiu candidatas com sorteio de atributos e sem esse sorteio. Também observamos 300 árvores nas trajetórias estendidas. Fixamos a escolha pela média do F1 macro nas 15 partes das sementes 42, 73 e 101.

| Variante | F1 na seleção | F1 médio nas conferências 509 e 911 | F1 na reserva |
|---|---:|---:|---:|
| Random Forest, sorteio de atributos | 89,73% | 89,38% | 82,63% |
| Bagging, todos os atributos por nó | 90,30% | 89,51% | 87,49% |

- **Random Forest:** 300 árvores; nove medidas; vazios pela mediana; mínimo de 2 por folha; profundidade sem limite fixo; apoio por folha; limite 50%.
- **Bagging:** 100 árvores; quatro medidas; vazios pela mediana; mínimo de 2 por folha; profundidade 6; apoio por folha; limite 50%.

Na Random Forest, sorteamos um subconjunto de atributos por nó. Na variante bagging, todos os atributos estão disponíveis e a diversidade vem do sorteio dos exemplos. Ambas usam a fórmula de média e desvio padrão no treinamento de cada parte.

As matrizes na reserva são `[[40, 10], [8, 46]]` para Random Forest e `[[44, 6], [7, 47]]` para bagging. A reserva e as novas divisões usam pessoas já observadas no projeto. Esses resultados não garantem a pontuação do Kaggle.

A indicação pela seleção foi **bagging**. O arquivo dessa indicação é `resultados/submissao_floresta_escolhida.csv`. Também geramos `submissao_random_forest.csv` e `submissao_bagging.csv`, cada um com 312 previsões. Nenhum desses arquivos foi enviado ao Kaggle nesta execução.

### Base e avaliação

Mantemos as 416 amostras para escolher a configuração e as mesmas 104 reservadas dos outros modelos. Cada paciente aparece uma vez. A busca inicial usa cinco partes da semente 42. Uma lista menor recebe mais duas divisões, 73 e 101; escolhemos pelo F1 macro médio dessas 15 partes.

Após fixar a escolha, conferimos as divisões 509 e 911 e outra semente da floresta. Depois observamos a reserva. Essas pessoas e essa reserva já foram usadas em outros experimentos do projeto; são comparações locais, e não um conjunto externo novo.

In [1]:
from pathlib import Path
from time import perf_counter
import csv
import hashlib
import json
import math
import random

import numpy as np

raiz = Path.cwd()
if not (raiz / "dados" / "treinamento_padronizado.csv").is_file():
    raiz = raiz.parent
caminho_base = raiz / "dados" / "treinamento_padronizado.csv"
with caminho_base.open(encoding="utf-8-sig", newline="") as arquivo:
    amostras = list(csv.DictReader(arquivo))

numericas = (
    "AGE", "HEIGHT", "WEIGHT", "SYSTOLIC", "DIASTOLIC", "GLUCOSE",
    "HEMOGLOBIN", "TRIGLYCERIDES", "SODIUM",
)
classes = np.array([int(a["class"]) for a in amostras], dtype=np.int8)
pacientes = [a["PATIENT_CODE"].strip() for a in amostras]
if any(not p for p in pacientes) or len(set(pacientes)) != len(pacientes):
    raise ValueError("Conferir os registros de paciente.")
if not np.all(np.isin(classes, [0, 1])):
    raise ValueError("Esperávamos classes 0 e 1.")

controle = np.flatnonzero(classes == 0).tolist()
diabetes = np.flatnonzero(classes == 1).tolist()
gerador = random.Random(42)
gerador.shuffle(controle)
gerador.shuffle(diabetes)
n0, n1 = round(len(controle) * 0.20), round(len(diabetes) * 0.20)
reservadas = np.array(controle[:n0] + diabetes[:n1])
treinamento = np.array(controle[n0:] + diabetes[n1:])


def dividir(indices, semente):
    partes = [[] for _ in range(5)]
    gerador = random.Random(semente)
    for classe in (0, 1):
        grupo = [int(i) for i in indices if classes[i] == classe]
        gerador.shuffle(grupo)
        for posicao, i in enumerate(grupo):
            partes[posicao % 5].append(i)
    for posicao, parte in enumerate(partes):
        referencia = [i for j, outra in enumerate(partes) if j != posicao for i in outra]
        yield np.array(referencia), np.array(parte)


def ler_numero(amostra, coluna):
    try:
        valor = float(str(amostra[coluna]).strip().replace(",", "."))
    except ValueError:
        return None
    if not math.isfinite(valor) or valor <= 0:
        return None
    if coluna == "HEMOGLOBIN" and valor > 100:
        return None
    return valor


print(f"Treinamento: {len(treinamento)} | Reserva: {len(reservadas)}")
print("SHA-256 da base:", hashlib.sha256(caminho_base.read_bytes()).hexdigest())

from itertools import combinations, product
import sys

print(f"Python {sys.version.split()[0]} | NumPy {np.__version__}")

Treinamento: 416 | Reserva: 104
SHA-256 da base: bc1a686d83c06e2d7667a6089e82b54b4bff2ca6467b81155f1246ac4719a032
Python 3.12.14 | NumPy 2.3.5


### Preparação dos atributos

Comparamos três exames, quatro medidas, nove medidas numéricas e as nove medidas com seis atributos categóricos. Para os vazios numéricos, comparamos média e mediana do treinamento.

Nas medidas numéricas, a média e o desvio usam os valores disponíveis. As categorias viram colunas de 0 e 1. Cada uma dessas colunas também recebe a fórmula de média e desvio. Uma coluna constante recebe zero. Categorias ausentes ou inéditas usam a categoria desconhecido.

Mantemos as decisões anteriores da base, incluindo a premissa sobre HbA1c e o tratamento de valores não positivos ou hemoglobina maior que 100 como ausentes. Essas são escolhas sobre os dados, não decisões aprendidas pela floresta.

In [2]:
categoricas = ("SMOKER", "FAMILY_HISTORY", "PHYSICAL_ACTIVITY", "GENDER", "BLOOD_GROUP", "RH_FACTOR")
grupos_rf = {
    "Três exames": ("GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES"),
    "Quatro medidas": ("SYSTOLIC", "GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES"),
    "Nove medidas": numericas,
    "Medidas e categorias": numericas + categoricas,
}


def ler_categoria(amostra, coluna):
    valor = amostra[coluna].strip()
    return "desconhecido" if valor in {"", "desconhecido"} or (coluna == "GENDER" and valor == "N") else valor


def preparar_rf(referencia, grupo, preenchimento):
    preparo = {"grupo": grupo, "preenchimento": preenchimento, "numeros": {}, "categorias": {}, "entradas": []}
    for coluna in grupos_rf[grupo]:
        if coluna in categoricas:
            categorias = sorted({ler_categoria(a, coluna) for a in referencia} | {"desconhecido"})
            regras = []
            for categoria in categorias:
                valores = np.array([float(ler_categoria(a, coluna) == categoria) for a in referencia])
                regras.append({"categoria": categoria, "media": float(valores.mean()), "desvio": float(valores.std(ddof=1))})
                preparo["entradas"].append(f"{coluna}__{categoria}")
            preparo["categorias"][coluna] = regras
        else:
            valores = [ler_numero(a, coluna) for a in referencia]
            disponiveis = np.array([v for v in valores if v is not None])
            if len(disponiveis) < 2:
                raise ValueError(f"Valores insuficientes em {coluna}.")
            estimativa = np.mean(disponiveis) if preenchimento == "media" else np.median(disponiveis)
            preparo["numeros"][coluna] = {"media": float(disponiveis.mean()), "desvio": float(disponiveis.std(ddof=1)), "estimativa": float(estimativa)}
            preparo["entradas"].append(coluna)
    return preparo


def transformar_rf(amostras_entrada, preparo):
    linhas = []
    for amostra in amostras_entrada:
        linha = []
        for coluna in grupos_rf[preparo["grupo"]]:
            if coluna in preparo["numeros"]:
                regra = preparo["numeros"][coluna]
                valor = ler_numero(amostra, coluna)
                if valor is None:
                    valor = regra["estimativa"]
                linha.append((valor - regra["media"]) / regra["desvio"] if regra["desvio"] else 0.0)
            else:
                regras = preparo["categorias"][coluna]
                categoria = ler_categoria(amostra, coluna)
                if categoria not in {r["categoria"] for r in regras}:
                    categoria = "desconhecido"
                for regra in regras:
                    valor = float(categoria == regra["categoria"])
                    linha.append((valor - regra["media"]) / regra["desvio"] if regra["desvio"] else 0.0)
        linhas.append(linha)
    return np.array(linhas, dtype=float)

### Como construímos cada árvore

Usamos entropia e ganho de informação para escolher o corte. Em cada nó, sorteamos atributos que têm variação naquele nó. Entre os candidatos, escolhemos o corte de maior ganho que respeite o mínimo de exemplos por folha. Se o primeiro subconjunto não tiver corte válido, examinamos os demais atributos disponíveis.

Paramos quando a folha é pura, quando chegamos à profundidade permitida ou quando não há corte de ganho positivo. Cada aparição de uma amostra no sorteio com reposição conta como um exemplo na árvore desta implementação.

Comparamos o sorteio pela raiz quadrada da quantidade de atributos com usar todos os atributos por nó. A segunda opção é a variante de árvores com amostragem por reposição, usualmente chamada bagging. A implementação mantém essas opções identificadas.

In [3]:
def entropia_rf(positivos, total):
    p = np.asarray(positivos, dtype=float) / total
    return -(p * np.log2(np.maximum(p, 1e-15)) + (1-p) * np.log2(np.maximum(1-p, 1e-15)))


def procurar_corte(x, y, atributos, minimo):
    if not len(atributos) or len(y) < 2 * minimo:
        return None
    valores = x[:, atributos]
    ordem = np.argsort(valores, axis=0, kind="stable")
    ordenados = np.take_along_axis(valores, ordem, axis=0)
    positivos = np.cumsum(y[ordem], axis=0)[:-1]
    n = np.arange(1, len(y))[:, None]
    validos = ((ordenados[:-1] < ordenados[1:]) & (n >= minimo) & (len(y)-n >= minimo))
    if not validos.any():
        return None
    depois = (n * entropia_rf(positivos, n) + (len(y)-n) * entropia_rf(y.sum()-positivos, len(y)-n)) / len(y)
    ganhos = float(entropia_rf(y.sum(), len(y))) - depois
    ganhos[~validos] = -np.inf
    atributo_local, posicao = divmod(int(np.argmax(ganhos.T)), len(y)-1)
    ganho = float(ganhos[posicao, atributo_local])
    if ganho <= 1e-12:
        return None
    limite = float((ordenados[posicao, atributo_local] + ordenados[posicao+1, atributo_local]) / 2)
    return int(atributos[atributo_local]), limite, ganho


def construir_arvore_rf(x, y, minimo, profundidade, quantidade_atributos, gerador, nivel=0):
    positivos = int(y.sum())
    no = {"amostras": len(y), "positivos": positivos, "classe": int(positivos * 2 > len(y))}
    if positivos in {0, len(y)} or len(y) < 2 * minimo or (profundidade is not None and nivel >= profundidade):
        return no
    disponiveis = np.flatnonzero(np.ptp(x, axis=0) > 0)
    if not len(disponiveis):
        return no
    quantidade = min(quantidade_atributos, len(disponiveis))
    candidatos = np.sort(gerador.choice(disponiveis, size=quantidade, replace=False))
    teste = procurar_corte(x, y, candidatos, minimo)
    if teste is None and quantidade < len(disponiveis):
        teste = procurar_corte(x, y, np.setdiff1d(disponiveis, candidatos), minimo)
    if teste is None:
        return no
    j, limite, ganho = teste
    esquerda = x[:, j] <= limite
    no.update(atributo=j, limite=limite, ganho=ganho)
    no["sim"] = construir_arvore_rf(x[esquerda], y[esquerda], minimo, profundidade, quantidade_atributos, gerador, nivel+1)
    no["nao"] = construir_arvore_rf(x[~esquerda], y[~esquerda], minimo, profundidade, quantidade_atributos, gerador, nivel+1)
    return no


def prever_arvore_rf(x, arvore):
    apoio = np.empty(len(x))
    voto = np.empty(len(x), dtype=np.int8)

    def percorrer(no, indices):
        if not len(indices):
            return
        if "atributo" not in no:
            apoio[indices] = no["positivos"] / no["amostras"]
            voto[indices] = no["classe"]
            return
        esquerda = x[indices, no["atributo"]] <= no["limite"]
        percorrer(no["sim"], indices[esquerda])
        percorrer(no["nao"], indices[~esquerda])

    percorrer(arvore, np.arange(len(x)))
    return apoio, voto

### Como combinamos as árvores

Comparamos duas formas de apoio: a fração de árvores que votam em diabetes e a média das proporções de diabetes nas folhas alcançadas. Uma maioria simples usa limite 0,5. Também comparamos limites 0,45 e 0,55 na validação.

As mesmas primeiras árvores são usadas quando observamos florestas de 30, 100 ou 300 árvores. Isso permite comparar o crescimento da floresta sem mudar todo o sorteio. O empate no limite indica controle. Esses apoios não foram calibrados como probabilidades clínicas.

In [4]:
def treinar_floresta(x, y, consulta, minimo, profundidade, atributos, pontos, semente=7):
    quantidade = max(1, int(np.sqrt(x.shape[1]))) if atributos == "sqrt" else x.shape[1]
    soma_apoios = np.zeros(len(consulta))
    soma_votos = np.zeros(len(consulta))
    floresta = []
    respostas = {}
    for numero in range(1, max(pontos)+1):
        gerador = np.random.default_rng(np.random.SeedSequence([semente, numero]))
        sorteados = gerador.integers(0, len(x), size=len(x))
        arvore = construir_arvore_rf(x[sorteados], y[sorteados], minimo, profundidade, quantidade, gerador)
        apoio, voto = prever_arvore_rf(consulta, arvore)
        soma_apoios += apoio
        soma_votos += voto
        floresta.append(arvore)
        if numero in pontos:
            respostas[numero] = {"folha": soma_apoios / numero, "voto": soma_votos / numero}
    return respostas, floresta


def decidir_rf(apoios, limite):
    return ((apoios > limite) & ~np.isclose(apoios, limite, rtol=0, atol=1e-12)).astype(np.int8)


def dados_rf(ref, aval, grupo, preenchimento):
    referencia = [amostras[i] for i in ref]
    consulta = [amostras[i] for i in aval]
    preparo = preparar_rf(referencia, grupo, preenchimento)
    return transformar_rf(referencia, preparo), transformar_rf(consulta, preparo), preparo

In [5]:
def contar(reais, previstas):
    return np.array([
        np.sum((reais == 0) & (previstas == 0)),
        np.sum((reais == 0) & (previstas == 1)),
        np.sum((reais == 1) & (previstas == 0)),
        np.sum((reais == 1) & (previstas == 1)),
    ])


def razao(a, b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)


def metricas(contagens):
    tn, fp, fn, tp = np.moveaxis(np.asarray(contagens), -1, 0)
    return {
        "acuracia": razao(tn + tp, tn + fp + fn + tp),
        "f1": (razao(2 * tn, 2 * tn + fp + fn) + razao(2 * tp, 2 * tp + fp + fn)) / 2,
        "precisao": (razao(tn, tn + fn) + razao(tp, tp + fp)) / 2,
        "recall0": razao(tn, tn + fp),
        "recall1": razao(tp, tp + fn),
    }


def mostrar(nome, contagens):
    m = metricas(contagens)
    print(f"{nome:24} | " + " | ".join(
        f"{np.mean(m[c]):>8.2%}" for c in ("acuracia", "f1", "precisao", "recall0", "recall1")
    ))

### Busca inicial

Comparamos quatro grupos de atributos, duas estimativas para os vazios, mínimo de duas ou oito amostras por folha, profundidade 6 ou sem limite fixo e dois limites de atributos por nó. São 64 trajetórias de treinamento.

Em cada trajetória observamos 30 e 100 árvores, as duas formas de apoio e três limites de decisão. São 768 configurações iniciais, comparadas nas mesmas cinco partes da semente 42. A semente da floresta é 7.

In [6]:
trajetos = list(product(grupos_rf, ("mediana", "media"), (2, 8), (6, None), ("sqrt", "todos")))
configuracoes = [t + (n, modo, limite) for t in trajetos for n, modo, limite in product((30, 100), ("voto", "folha"), (0.45, 0.5, 0.55))]
rodadas = []
previsoes_cv = np.full((len(treinamento), len(configuracoes)), -1, dtype=np.int8)
posicao_treinamento = {int(i): j for j, i in enumerate(treinamento)}
inicio = perf_counter()
for numero, (ref, aval) in enumerate(dividir(treinamento, 42), start=1):
    cache_dados = {}
    resultados = []
    posicoes = [posicao_treinamento[int(i)] for i in aval]
    for indice_trajeto, trajeto in enumerate(trajetos):
        grupo, preenchimento, minimo, profundidade, atributos = trajeto
        chave = grupo, preenchimento
        if chave not in cache_dados:
            cache_dados[chave] = dados_rf(ref, aval, *chave)
        x, consulta, _ = cache_dados[chave]
        respostas, _ = treinar_floresta(x, classes[ref], consulta, minimo, profundidade, atributos, (30, 100))
        for n, modo, limite in product((30, 100), ("voto", "folha"), (0.45, 0.5, 0.55)):
            previstas = decidir_rf(respostas[n][modo], limite)
            coluna = len(resultados)
            previsoes_cv[posicoes, coluna] = previstas
            resultados.append(contar(classes[aval], previstas))
        if (indice_trajeto+1) % 16 == 0:
            print(f"Parte {numero}: {indice_trajeto+1}/{len(trajetos)} trajetórias", flush=True)
    rodadas.append(resultados)
rodadas = np.array(rodadas)
f1_inicial = metricas(rodadas)["f1"].mean(axis=0)


def chave_rf(i, valores, grade):
    grupo, preenchimento, minimo, profundidade, atributos, n, modo, limite = grade[i]
    return (-float(valores[i]), len(grupos_rf[grupo]), n, profundidade if profundidade is not None else 999,
            -minimo, modo != "voto", abs(limite-0.5), i)


ordem = sorted(range(len(configuracoes)), key=lambda i: chave_rf(i, f1_inicial, configuracoes))
lista = []
assinaturas = set()
for i in ordem:
    assinatura = np.packbits(previsoes_cv[:, i]).tobytes()
    if assinatura not in assinaturas:
        lista.append(i)
        assinaturas.add(assinatura)
    if len(lista) == 8:
        break
for grupo in grupos_rf:
    i = next(i for i in ordem if configuracoes[i][0] == grupo)
    if i not in lista:
        lista.append(i)
assinaturas_sqrt = set()
adicionados_sqrt = 0
for i in ordem:
    if configuracoes[i][4] != "sqrt":
        continue
    assinatura = np.packbits(previsoes_cv[:, i]).tobytes()
    if assinatura in assinaturas_sqrt:
        continue
    assinaturas_sqrt.add(assinatura)
    if i not in lista:
        lista.append(i)
    adicionados_sqrt += 1
    if adicionados_sqrt == 4:
        break

trajetos_extensao = []
for i in ordem:
    trajeto = configuracoes[i][:5]
    if trajeto not in trajetos_extensao:
        trajetos_extensao.append(trajeto)
    if len(trajetos_extensao) == 4:
        break
melhor_sorteio = next(i for i in ordem if configuracoes[i][4] == "sqrt")
if configuracoes[melhor_sorteio][:5] not in trajetos_extensao:
    trajetos_extensao.append(configuracoes[melhor_sorteio][:5])
print("Melhor configuração com sorteio de atributos:", configuracoes[melhor_sorteio])
print(f"Busca inicial: {perf_counter()-inicio:.1f} s")
for i in ordem[:10]:
    print(f"F1 {f1_inicial[i]:.2%} | {configuracoes[i]}")
np.savez_compressed(raiz / "resultados" / "random_forest_busca_inicial.npz",
    catalogo=json.dumps({"configuracoes": configuracoes, "lista": lista, "trajetos_extensao": trajetos_extensao,
                        "sha_base": hashlib.sha256(caminho_base.read_bytes()).hexdigest()}, ensure_ascii=False),
    rodadas=rodadas, previsoes_cv=previsoes_cv)

Parte 1: 16/64 trajetórias
Parte 1: 32/64 trajetórias
Parte 1: 48/64 trajetórias
Parte 1: 64/64 trajetórias
Parte 2: 16/64 trajetórias
Parte 2: 32/64 trajetórias
Parte 2: 48/64 trajetórias
Parte 2: 64/64 trajetórias
Parte 3: 16/64 trajetórias
Parte 3: 32/64 trajetórias
Parte 3: 48/64 trajetórias
Parte 3: 64/64 trajetórias
Parte 4: 16/64 trajetórias
Parte 4: 32/64 trajetórias
Parte 4: 48/64 trajetórias
Parte 4: 64/64 trajetórias
Parte 5: 16/64 trajetórias
Parte 5: 32/64 trajetórias
Parte 5: 48/64 trajetórias
Parte 5: 64/64 trajetórias
Busca inicial: 104.2 s
F1 90.60% | ('Quatro medidas', 'media', 8, 6, 'todos', 30, 'voto', 0.45)
F1 90.37% | ('Medidas e categorias', 'media', 8, None, 'todos', 30, 'voto', 0.55)
F1 90.37% | ('Nove medidas', 'media', 8, None, 'todos', 30, 'voto', 0.55)
F1 90.37% | ('Quatro medidas', 'mediana', 2, 6, 'todos', 100, 'folha', 0.5)
F1 90.36% | ('Medidas e categorias', 'mediana', 8, None, 'todos', 30, 'folha', 0.5)
F1 90.36% | ('Quatro medidas', 'mediana', 8, Non

Segunda seleção inclui quatro configurações com sorteio de atributos. Trajetórias estendidas: 5.


### Segunda seleção e crescimento da floresta

Levamos configurações de destaque à segunda comparação, incluindo o melhor grupo de cada conjunto de atributos. Incluímos quatro configurações de destaque com sorteio de atributos. Estendemos para 300 árvores as quatro trajetórias mais bem avaliadas e a melhor com sorteio, caso ela ainda não esteja nesse grupo.

Usamos as sementes 42, 73 e 101 e escolhemos pelo F1 macro médio nas 15 partes. As contagens da primeira busca são reaproveitadas para as configurações já avaliadas na semente 42. Empates favorecem menos atributos, menos árvores e menor profundidade.

In [7]:
extensoes = [t + (300, modo, limite) for t in trajetos_extensao for modo, limite in product(("voto", "folha"), (0.45, 0.5, 0.55))]
candidatas = [configuracoes[i] for i in lista] + extensoes
mapa_inicial = {config: i for i, config in enumerate(configuracoes)}


def avaliar_florestas(configs, semente_divisao, semente_floresta=7):
    resultados = np.zeros((5, len(configs), 4), dtype=np.int16)
    for numero, (ref, aval) in enumerate(dividir(treinamento, semente_divisao)):
        pendentes = {}
        for i, config in enumerate(configs):
            if semente_divisao == 42 and semente_floresta == 7 and config in mapa_inicial:
                resultados[numero, i] = rodadas[numero, mapa_inicial[config]]
            else:
                pendentes.setdefault(config[:5], []).append(i)
        for trajeto, indices in pendentes.items():
            grupo, preenchimento, minimo, profundidade, atributos = trajeto
            pontos = sorted({configs[i][5] for i in indices})
            x, consulta, _ = dados_rf(ref, aval, grupo, preenchimento)
            respostas, _ = treinar_floresta(x, classes[ref], consulta, minimo, profundidade, atributos, pontos, semente_floresta)
            for i in indices:
                n, modo, limite = configs[i][5:]
                resultados[numero, i] = contar(classes[aval], decidir_rf(respostas[n][modo], limite))
        print(f"Segunda avaliação: divisão {semente_divisao}, floresta {semente_floresta}, parte {numero+1}", flush=True)
    return resultados


selecao = np.concatenate([avaliar_florestas(candidatas, s) for s in (42, 73, 101)], axis=0)
f1_selecao = metricas(selecao)["f1"].mean(axis=0)
ordem_final = sorted(range(len(candidatas)), key=lambda i: chave_rf(i, f1_selecao, candidatas))
escolhida = ordem_final[0]
configuracao_final = candidatas[escolhida]
melhor_rf = next(i for i in ordem_final if candidatas[i][4] == "sqrt")
melhor_bagging = next(i for i in ordem_final if candidatas[i][4] == "todos")
configuracao_rf = candidatas[melhor_rf]
configuracao_bagging = candidatas[melhor_bagging]
print("Escolhida no geral:", configuracao_final)
print("Random Forest com sorteio:", configuracao_rf)
mostrar("RF com sorteio, seleção", selecao[:, melhor_rf])
print("Bagging:", configuracao_bagging)
mostrar("Bagging, seleção", selecao[:, melhor_bagging])
mostrar("Melhor conjunto, seleção", selecao[:, escolhida])
np.savez_compressed(raiz / "resultados" / "random_forest_selecao.npz",
    catalogo=json.dumps({"candidatas": candidatas, "sementes": [42, 73, 101], "semente_floresta": 7,
                        "grupos": grupos_rf, "sha_base": hashlib.sha256(caminho_base.read_bytes()).hexdigest()}, ensure_ascii=False),
    selecao=selecao, escolhida=escolhida, melhor_rf=melhor_rf, melhor_bagging=melhor_bagging)

Segunda avaliação: divisão 42, floresta 7, parte 1
Segunda avaliação: divisão 42, floresta 7, parte 2
Segunda avaliação: divisão 42, floresta 7, parte 3
Segunda avaliação: divisão 42, floresta 7, parte 4
Segunda avaliação: divisão 42, floresta 7, parte 5
Segunda avaliação: divisão 73, floresta 7, parte 1
Segunda avaliação: divisão 73, floresta 7, parte 2
Segunda avaliação: divisão 73, floresta 7, parte 3
Segunda avaliação: divisão 73, floresta 7, parte 4
Segunda avaliação: divisão 73, floresta 7, parte 5
Segunda avaliação: divisão 101, floresta 7, parte 1
Segunda avaliação: divisão 101, floresta 7, parte 2
Segunda avaliação: divisão 101, floresta 7, parte 3
Segunda avaliação: divisão 101, floresta 7, parte 4
Segunda avaliação: divisão 101, floresta 7, parte 5
Escolhida no geral: ('Quatro medidas', 'mediana', 2, 6, 'todos', 100, 'folha', 0.5)
Random Forest com sorteio: ('Nove medidas', 'mediana', 2, None, 'sqrt', 300, 'folha', 0.5)
RF com sorteio, seleção  |   89.75% |   89.73% |   89.9

### Conferências e reserva

Fixamos a configuração antes de observar as divisões 509 e 911, e antes de mudar a semente da floresta para 97 na divisão 42. Essas conferências acompanham a variação dos dados e dos sorteios das árvores.

Comparamos a melhor Random Forest com sorteio de atributos e a melhor variante bagging. Depois treinamos cada uma nas 416 amostras e avaliamos nas 104 reservadas. A comparação recupera também os resultados dos componentes com média e desvio padrão. A reserva já foi consultada no projeto e não escolhe novos parâmetros nesta etapa.

In [8]:
variantes = [("Random Forest", configuracao_rf), ("Bagging", configuracao_bagging)]
conferencias = []
for divisao, semente_floresta in ((509, 7), (911, 7), (42, 97)):
    resultado = avaliar_florestas([c for _, c in variantes], divisao, semente_floresta)
    conferencias.append(resultado)
    for j, (nome, _) in enumerate(variantes):
        mostrar(f"{nome}, divisão {divisao}, floresta {semente_floresta}", resultado[:, j])

contagens_reserva = []
previsoes_reserva = []
for nome, config in variantes:
    grupo, preenchimento, minimo, profundidade, atributos, n, modo, limite = config
    x, consulta, preparo_reserva = dados_rf(treinamento, reservadas, grupo, preenchimento)
    respostas, _ = treinar_floresta(x, classes[treinamento], consulta, minimo, profundidade, atributos, (n,))
    previstas = decidir_rf(respostas[n][modo], limite)
    reserva = contar(classes[reservadas], previstas)
    contagens_reserva.append(reserva)
    previsoes_reserva.append(previstas)
    mostrar(nome + ", reserva", reserva)
    print("Matriz:", reserva.reshape(2, 2).tolist())
    print("Acertos:", int(reserva[0]+reserva[3]), "de", len(reservadas))

caminho_componentes = raiz / "resultados" / "combinacoes_zscore_cache.npz"
if caminho_componentes.exists():
    with np.load(caminho_componentes, allow_pickle=False) as componentes:
        if componentes["sha_base"].item() != hashlib.sha256(caminho_base.read_bytes()).hexdigest():
            raise ValueError("Base dos componentes diferente.")
        for j, nome in enumerate(("KNN, escala padrão", "Árvore, escala padrão", "Rede, escala padrão")):
            mostrar(nome, contar(classes[reservadas], componentes["rotulos_reserva"][:, j]))
np.savez_compressed(raiz / "resultados" / "random_forest_conferencias.npz",
    conferencias=np.array(conferencias), reserva=np.array(contagens_reserva),
    previstas_reserva=np.array(previsoes_reserva), variantes=np.array([n for n, _ in variantes]))


Segunda avaliação: divisão 509, floresta 7, parte 1
Segunda avaliação: divisão 509, floresta 7, parte 2
Segunda avaliação: divisão 509, floresta 7, parte 3
Segunda avaliação: divisão 509, floresta 7, parte 4
Segunda avaliação: divisão 509, floresta 7, parte 5
Random Forest, divisão 509, floresta 7 |   89.42% |   89.38% |   89.65% |   89.00% |   89.80%
Bagging, divisão 509, floresta 7 |   88.94% |   88.92% |   89.27% |   91.50% |   86.54%
Segunda avaliação: divisão 911, floresta 7, parte 1
Segunda avaliação: divisão 911, floresta 7, parte 2
Segunda avaliação: divisão 911, floresta 7, parte 3
Segunda avaliação: divisão 911, floresta 7, parte 4
Segunda avaliação: divisão 911, floresta 7, parte 5
Random Forest, divisão 911, floresta 7 |   89.41% |   89.37% |   89.63% |   87.50% |   91.21%
Bagging, divisão 911, floresta 7 |   90.13% |   90.10% |   90.44% |   90.00% |   90.25%
Segunda avaliação: divisão 42, floresta 97, parte 1
Segunda avaliação: divisão 42, floresta 97, parte 2
Segunda aval

### Treinamento final e arquivo para o Kaggle

Treinamos as duas variantes com todas as 520 amostras e identificamos a indicação pelo F1 macro da seleção. O teste recebe os parâmetros de preparação desse treinamento completo. Salvamos a preparação, a configuração e as árvores, além do CSV de 312 previsões.

Reabrimos o modelo e conferimos se ele reproduz o CSV. A pontuação no Kaggle ainda depende do envio; a configuração fica fixada pelo critério local.

In [9]:
def padronizar_teste(amostra):
    entrada = amostra.copy()
    if entrada["AGE"].strip() in {"0", "-1", "999"}:
        entrada["AGE"] = ""
    altura = ler_numero(entrada, "HEIGHT")
    if altura is not None and altura < 3:
        entrada["HEIGHT"] = str(altura * 100)
    pressao = entrada["BLOOD_PRESSURE"].strip().replace("x", "/").replace(" ", "")
    if pressao in {"", "?"}:
        entrada["SYSTOLIC"], entrada["DIASTOLIC"] = "", ""
    else:
        sistolica, diastolica = pressao.split("/")
        entrada["SYSTOLIC"] = str(int(sistolica))
        entrada["DIASTOLIC"] = str(int(diastolica))

    genero = entrada["GENDER"].strip().upper()
    entrada["GENDER"] = {"MALE": "M", "M": "M", "FEMALE": "F", "F": "F", "": "N"}.get(genero, genero)
    mapas = {
        "SMOKER": {"never": "nunca_fumou", "former": "ex_fumante", "ex-smoker": "ex_fumante", "current": "fumante_atual", "yes": "fumante_atual", "": ""},
        "PHYSICAL_ACTIVITY": {"sedentary": "sedentario", "sedentry": "sedentario", "light": "leve", "ligth": "leve", "moderate": "moderada", "moderat": "moderada", "intense": "intensa", "intensive": "intensa", "": ""},
        "FAMILY_HISTORY": {"yes": "sim", "y": "sim", "sim": "sim", "1": "sim", "no": "nao", "n": "nao", "não": "nao", "nao": "nao", "0": "nao", "unknown": "desconhecido", "desconhecido": "desconhecido", "": "desconhecido"},
    }
    for coluna, mapa in mapas.items():
        valor = entrada[coluna].strip().lower()
        entrada[coluna] = mapa.get(valor, valor)
    sangue = entrada["BLOOD_TYPE"].strip()
    entrada["BLOOD_GROUP"] = sangue[:-1] if sangue else ""
    entrada["RH_FACTOR"] = sangue[-1] if sangue else ""
    return entrada



with (raiz / "dados" / "kaggle-test" / "test.csv").open(encoding="utf-8-sig", newline="") as arquivo:
    teste = [padronizar_teste(a) for a in csv.DictReader(arquivo)]
ids_teste = [a["id"] for a in teste]
assert len(ids_teste) == len(set(ids_teste)) == 312
arquivos_variantes = [("Random Forest", configuracao_rf, "random_forest"),
                       ("Bagging", configuracao_bagging, "bagging")]
for nome, config, prefixo_arquivo in arquivos_variantes:
    grupo, preenchimento, minimo, profundidade, atributos, n, modo, limite = config
    preparo_final = preparar_rf(amostras, grupo, preenchimento)
    x_final = transformar_rf(amostras, preparo_final)
    x_teste = transformar_rf(teste, preparo_final)
    respostas, floresta_final = treinar_floresta(x_final, classes, x_teste, minimo, profundidade, atributos, (n,))
    previstas_teste = decidir_rf(respostas[n][modo], limite)
    caminho_modelo = raiz / "resultados" / f"{prefixo_arquivo}_modelo.json"
    modelo = {"configuracao": config, "preparo": preparo_final, "arvores": floresta_final,
              "semente": 7, "variante": nome, "sha_base": hashlib.sha256(caminho_base.read_bytes()).hexdigest()}
    caminho_modelo.write_text(json.dumps(modelo, ensure_ascii=False) + "\n", encoding="utf-8")
    caminho_csv = raiz / "resultados" / f"submissao_{prefixo_arquivo}.csv"
    with caminho_csv.open("w", encoding="utf-8", newline="") as arquivo:
        escritor = csv.writer(arquivo)
        escritor.writerow(["id", "class"])
        escritor.writerows(zip(ids_teste, previstas_teste.tolist(), strict=True))

    reaberto = json.loads(caminho_modelo.read_text(encoding="utf-8"))
    x_reaberto = transformar_rf(teste, reaberto["preparo"])
    apoios_reabertos = []
    for arvore in reaberto["arvores"]:
        apoio, voto = prever_arvore_rf(x_reaberto, arvore)
        apoios_reabertos.append(voto if modo == "voto" else apoio)
    previsoes_reabertas = decidir_rf(np.mean(apoios_reabertos, axis=0), limite)
    assert np.array_equal(previsoes_reabertas, previstas_teste)
    with caminho_csv.open(encoding="utf-8", newline="") as arquivo:
        leitor = csv.DictReader(arquivo)
        assert leitor.fieldnames == ["id", "class"]
        linhas = list(leitor)
    assert len(linhas) == 312
    assert [a["id"] for a in linhas] == ids_teste
    assert all(a["class"] in {"0", "1"} for a in linhas)
    print("Variante:", nome, "| CSV:", caminho_csv.relative_to(raiz))
    print("Configuração:", config)
    print("Entradas:", len(preparo_final["entradas"]))
    print("SHA-256 do CSV:", hashlib.sha256(caminho_csv.read_bytes()).hexdigest())
    print("Modelo reaberto reproduz as 312 previsões.")

variante_escolhida = "random_forest" if configuracao_final[4] == "sqrt" else "bagging"
escolha_modelo = {"variante": variante_escolhida, "configuracao": configuracao_final,
                  "f1_selecao": float(f1_selecao[escolhida]), "criterio": "F1 macro médio nas 15 partes da seleção"}
(raiz / "resultados" / "floresta_escolhida.json").write_text(json.dumps(escolha_modelo, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(raiz / "resultados" / "submissao_floresta_escolhida.csv").write_bytes(
    (raiz / "resultados" / f"submissao_{variante_escolhida}.csv").read_bytes())
print("Indicação pela seleção:", variante_escolhida)


Variante: Random Forest | CSV: resultados\submissao_random_forest.csv
Configuração: ('Nove medidas', 'mediana', 2, None, 'sqrt', 300, 'folha', 0.5)
Entradas: 9
SHA-256 do CSV: a1905e216c32ac652dd12f9ad838cb51dce173818b1777ee6f9cf0a247d86620
Modelo reaberto reproduz as 312 previsões.
Variante: Bagging | CSV: resultados\submissao_bagging.csv
Configuração: ('Quatro medidas', 'mediana', 2, 6, 'todos', 100, 'folha', 0.5)
Entradas: 4
SHA-256 do CSV: d95f76f51bb62225cd2c1e23d9b0c7cb37cca44451901d39dc808dd9f5b79c1a
Modelo reaberto reproduz as 312 previsões.
Indicação pela seleção: bagging


### Referências

- [Documentação técnica de Random Forest](https://scikit-learn.org/stable/modules/ensemble.html#forest): amostragem com reposição, sorteio de atributos e combinação das árvores. Usamos essa descrição como referência conceitual para a implementação própria.
- [Parâmetros da RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html): significado dos parâmetros comparados.
- [Aula 5](../docs/materiais_aula/aula5_MD.pdf): árvores, entropia e ganho de informação.
- [Aula 7](../docs/materiais_aula/aula7_MD.pdf): avaliação dos classificadores.
- [Aula 10](../docs/materiais_aula/aula10_MD.pdf): normalização pela média e desvio padrão.

O sorteio da floresta, os critérios de parada, as sementes, os grupos e a grade de configurações são decisões desta aplicação. A implementação está incluída no notebook para leitura e reprodução.